# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [1]:
from class_dataset import SegmentationClassDataset, num_classes as class_num_classes
from instance_dataset import SegmentationInstanceDataset

# mode = "instance"
mode = "class"

if mode == "instance":
    dataset_class = SegmentationInstanceDataset
    num_classes = 2
    model_path = "model_instance.pth"
elif mode == "class":
    dataset_class = SegmentationClassDataset
    num_classes = class_num_classes
    model_path = "model_class.pth"

In [2]:
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils

import torch


model = get_model_instance_segmentation(num_classes)

dataset = dataset_class("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

{'loss_classifier': tensor(1.7098, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.6715, grad_fn=<DivBackward0>), 'loss_objectness': tensor(2.4257, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.4597, grad_fn=<DivBackward0>)}
{'boxes': tensor([[3.0040e+01, 1.8315e+01, 3.9213e+02, 2.0563e+02],
        [2.5451e+02, 0.0000e+00, 3.9724e+02, 1.6443e+02],
        [2.4458e+02, 1.0155e+02, 3.9868e+02, 2.3174e+02],
        [1.4343e+02, 9.1268e+01, 3.9129e+02, 2.5446e+02],
        [0.0000e+00, 5.7785e+00, 1.5958e+02, 1.6731e+02],
        [2.9349e+02, 2.0758e+01, 3.9928e+02, 2.3454e+02],
        [4.2642e+01, 1.2765e+02, 2.1168e+02, 2.5714e+02],
        [2.7853e+01, 1.8230e+02, 2.2072e+02, 2.6671e+02],
        [1.7214e+02, 1.5772e+02, 3.8857e+02, 2.4281e+02],
        [0.0000e+00, 1.2486e+02, 1.4479e+02, 2.5585e+02],
        [2.3960e+02, 1.2344e+01, 3.7245e+02, 2.4747e+02],
        [3.4520e+02, 0.0000e+00, 4.0000e+02, 2.8435e+02],
        [1.7534e+02, 4.3689e+0

## Training and Validation


In [3]:
import csv
from engine import evaluate, train_one_epoch
from sklearn.model_selection import KFold


k_folds = 5
num_epochs = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")

kfold = KFold(n_splits=k_folds, shuffle=True)

# use our dataset and defined transformations
full_dataset = dataset_class("../../training_data", get_transforms(is_train=True))

results = [
    [
        "fold",
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):
    print(f"FOLD {fold}")
    print("-" * 20)

    train_dataset = torch.utils.data.Subset(full_dataset, train_ids)
    train_dataset.dataset.transforms = get_transforms(is_train=True)

    valid_dataset = torch.utils.data.Subset(full_dataset, test_ids)
    valid_dataset.dataset.transforms = get_transforms(is_train=False)

    # define training and validation data loaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=1, shuffle=True, collate_fn=utils.collate_fn
    )
    validation_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=1, shuffle=False, collate_fn=utils.collate_fn
    )

    model = get_model_instance_segmentation(num_classes).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        training_losses = train_one_epoch(
            model, optimizer, train_loader, device, epoch, print_freq=10
        )

        for batch, losses in enumerate(training_losses):
            results.append(
                [
                    fold,
                    epoch,
                    batch,
                    losses["loss_classifier"],
                    losses["loss_box_reg"],
                    losses["loss_objectness"],
                    losses["loss_rpn_box_reg"],
                ]
            )

        # update the learning rate
        lr_scheduler.step()
        # evaluate on the validation dataset
        evaluate(model, validation_loader, device=device)

# save results to a CSV file
with open("results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

Using cpu for training.
FOLD 0
--------------------
Epoch: [0]  [ 0/96]  eta: 0:06:22  lr: 0.000058  loss: 1.7564 (1.7564)  loss_classifier: 1.6356 (1.6356)  loss_box_reg: 0.1024 (0.1024)  loss_objectness: 0.0182 (0.0182)  loss_rpn_box_reg: 0.0002 (0.0002)  time: 3.9803  data: 0.2004
Epoch: [0]  [10/96]  eta: 0:05:02  lr: 0.000583  loss: 3.1804 (3.5826)  loss_classifier: 1.5282 (1.4474)  loss_box_reg: 0.5707 (0.5587)  loss_objectness: 0.8971 (1.3240)  loss_rpn_box_reg: 0.2342 (0.2525)  time: 3.5202  data: 0.2646
Epoch: [0]  [20/96]  eta: 0:04:27  lr: 0.001109  loss: 1.9439 (2.5704)  loss_classifier: 0.9151 (0.9892)  loss_box_reg: 0.5209 (0.4768)  loss_objectness: 0.4451 (0.8531)  loss_rpn_box_reg: 0.1988 (0.2513)  time: 3.4913  data: 0.2746
Epoch: [0]  [30/96]  eta: 0:03:58  lr: 0.001635  loss: 1.2123 (2.0721)  loss_classifier: 0.3454 (0.7564)  loss_box_reg: 0.3817 (0.4586)  loss_objectness: 0.1963 (0.6549)  loss_rpn_box_reg: 0.0676 (0.2022)  time: 3.6616  data: 0.2773


KeyboardInterrupt: 

In [ ]:
# Train final model on all data

print(f"FINAL TRAINING")
print("-" * 20)

# define training and validation data loaders
train_loader = torch.utils.data.DataLoader(
    full_dataset, batch_size=1, shuffle=True, collate_fn=utils.collate_fn
)

model = get_model_instance_segmentation(num_classes).to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

results = [
    [
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    training_losses = train_one_epoch(
        model, optimizer, train_loader, device, epoch, print_freq=10
    )

    for batch, losses in enumerate(training_losses):
        results.append(
            [
                epoch,
                batch,
                losses["loss_classifier"],
                losses["loss_box_reg"],
                losses["loss_objectness"],
                losses["loss_rpn_box_reg"],
            ]
        )

    # update the learning rate
    lr_scheduler.step()

# save results to a CSV file
with open("final-results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

# save the model
torch.save(model.state_dict(), model_path)